In [1]:
platformID = 'FBE'


In [2]:
from datetime import datetime
import pandas as pd

import os 

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from functions import execute_sql_query, compare_or_update_reference
import test_functions

from config import gam_info

In [13]:
# country
country_cols = ['YT-_FBE_codes', 'PlaceID']
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID',
                             keep_default_na=False)[country_cols]
# week 
week_cols = ['w/c']
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

today = pd.Timestamp.today().normalize()
last_monday = today - pd.Timedelta(days=(today.weekday() % 7))
week_tester = week_tester[week_tester['w/c'] < last_monday]

# social media accoutns
channel_cols=['Channel ID']
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

#socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = platformID + socialmedia_accounts['Channel ID']
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
test_functions.test_lookup_files(country_codes, country_cols, [f"{platformID}_1c_0", f"{platformID}_1c_1", f"{platformID}_1c_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_1c_3", f"{platformID}_1c_4", f"{platformID}_1c_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_1c_6", f"{platformID}_1c_7", f"{platformID}_1c_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test FBE_1c_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_2 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_5 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_8 passed: No missing values in lookup.
...updating logbook...



## country

In [14]:

def reconstruct_ids_from_truncation(
    messed_ids,
    original_ids,
    *,
    min_prefix: int = 6,
    max_prefix: int = 12,
    fallback_prefix: int = 5,
    min_score_threshold: float = 0.0
):
    """
    Reconstruct actual IDs for truncated/rounded numeric-string IDs by prefix matching
    against canonical/original IDs.

    Parameters
    ----------
    messed_ids : list-like or pandas Series of str/int
        The truncated IDs (some may already be intact).
    original_ids : list-like or pandas Series of str/int
        The canonical/original IDs to match against.
    min_prefix : int, optional
        Minimum prefix length to index and compare (default: 6).
    max_prefix : int, optional
        Maximum prefix length to index and compare (default: 12).
    fallback_prefix : int, optional
        Prefix length to use when no exact prefix-bucket hit (default: 5).
    min_score_threshold : float, optional
        Minimum score to accept a match (raise to be stricter, e.g., 6.0).

    Returns
    -------
    pandas.DataFrame
        Columns:
          - messed_id (str)
          - messed_base (str)
          - trailing_zeros (int)
          - reconstructed_id (str or None)
          - match_score (float or None)
          - matched (bool)
    """
    import re
    from collections import defaultdict
    import pandas as pd

    # --- helpers ---
    def trailing_zeros(s: str) -> int:
        m = re.search(r'0+$', s)
        return len(m.group(0)) if m else 0

    def normalize_messed_id(s: str) -> str:
        # strip trailing zeros; keep as string; avoid None
        return s.rstrip('0')

    def build_prefix_index(original_ids, min_prefix=6, max_prefix=12):
        idx = defaultdict(list)
        for s in original_ids:
            s = str(s)
            for k in range(min_prefix, min(max_prefix, len(s)) + 1):
                idx[(k, s[:k])].append(s)
        return idx

    def score_candidate(base: str, candidate: str) -> float:
        # common prefix length first
        common = 0
        for a, b in zip(base, candidate):
            if a == b:
                common += 1
            else:
                break
        # soft length penalty (longer difference -> slight penalty)
        length_pen = abs(len(candidate) - len(base)) * 0.1
        # small boost if first 6 digits match (helps tie-breaking)
        boost = 0.5 if len(base) >= 6 and candidate.startswith(base[:6]) else 0.0
        return common - length_pen + boost

    originals = [str(x) for x in original_ids]
    messed = [str(x) for x in messed_ids]

    # Fast path: if some messed IDs are already exact originals, map directly
    originals_set = set(originals)

    prefix_index = build_prefix_index(originals, min_prefix=min_prefix, max_prefix=max_prefix)

    rows = []
    for m in messed:
        # If already exact, keep it
        if m in originals_set:
            rows.append({
                "messed_id": m,
                "messed_base": m,
                "trailing_zeros": 0,
                "reconstructed_id": m,
                "match_score": float("inf"),
                "matched": True,
            })
            continue

        base = normalize_messed_id(m)
        tz = trailing_zeros(m)
        found = None
        best_score = float("-inf")

        if base:  # ignore empty base edge cases like '0'/'000...'
            # Try descending prefix lengths (specific -> general)
            for k in range(min(max_prefix, len(base)), min_prefix - 1, -1):
                bucket = prefix_index.get((k, base[:k]), [])
                if bucket:
                    cand = max(bucket, key=lambda c: score_candidate(base, c))
                    sc = score_candidate(base, cand)
                    if sc > best_score:
                        best_score = sc
                        found = cand
                    # heuristic early exit: very strong match
                    if best_score >= k - 0.5:
                        break

            # Fallback: broad filter on first N digits and choose best by score
            if not found and len(base) >= fallback_prefix:
                cands = [s for s in originals if s.startswith(base[:fallback_prefix])]
                if cands:
                    cand = max(cands, key=lambda c: score_candidate(base, c))
                    sc = score_candidate(base, cand)
                    if sc > best_score:
                        best_score = sc
                        found = cand

        matched = bool(found) and best_score >= min_score_threshold

        rows.append({
            "messed_id": m,
            "messed_base": base,
            "trailing_zeros": tz,
            "reconstructed_id": found if matched else None,
            "match_score": best_score if matched else None,
            "matched": matched,
        })

    return pd.DataFrame(rows)


In [17]:
sql_query = f"""
    SELECT 
        week_commencing,
        page_id ,
        page_name,
        page_fans_country_total AS page_fans_country,
        country_code AS fb_metric_breakdown
    FROM
        redshiftdb.central_insights.adverity_social_facebook_page_fans_by_country
    WHERE
        week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"
'''
df = execute_sql_query(sql_query)
df['page_id'] = platformID+df['page_id'].astype(str)
df.to_csv(file, index=False, na_rep='')
'''
try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+facebook_country_raw['page_id']
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

facebook_country_raw = pd.read_csv(file, keep_default_na=False, dtype={"page_id": "string"}).drop_duplicates()
facebook_country_raw['week_commencing'] = pd.to_datetime(facebook_country_raw['week_commencing'])
facebook_country_raw = facebook_country_raw.rename(columns={'page_id': 'Channel ID',
                                                            'page_name': 'Channel Name',
                                                            'week_commencing': 'w/c',
                                                            'fb_metric_breakdown': 'YT-_FBE_codes'})



no redshift connection using last time queried


In [18]:
### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_country_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_9",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               id_column='Channel ID',
                                               main_data=facebook_country_raw,
                                               week_lookup=week_tester[['w/c']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_country_raw, 
                           numeric_columns=['page_fans_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_country_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
❌ Test FBE_1c_9 failed: not all elements from lookup were retrieved in dataset - decide if they are really missing or if these accounts are inactive 
...updating logbook...

❌ Test FBE_1c_10 failed: Missing completed weeks detected.
...updating logbook...



/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  week_lookup[key] = pd.to_datetime(week_lookup[key])


✅ Test FBE_1c_11 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test FBE_1c_12 passed: No combinations occurs more than once a week.
...updating logbook...



In [19]:
# filter to relevant pageID's
facebook_country = facebook_country_raw[facebook_country_raw['Channel ID'].isin(channel_ids)]
test_functions.test_inner_join(facebook_country, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='calculating country %')

facebook_country = facebook_country.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])



Inner join test FBE_1c_13 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...



In [20]:

# Group by specified columns and sum the fb_metric_value
facebook_country_sum = facebook_country.groupby(['Channel ID', 'w/c'])\
                                        .agg(Sum_page_fans_country=('page_fans_country', 'sum'))\
                                        .reset_index()
facebook_country = facebook_country.merge(facebook_country_sum, 
                                          how='inner', on= ['Channel ID', 'w/c'])
test_functions.test_inner_join(facebook_country, facebook_country_sum, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_14", 
                               test_step='calculating country %')

facebook_country['country_%'] = facebook_country['page_fans_country']/facebook_country['Sum_page_fans_country']

### RUN TESTS
test_functions.test_percentage(facebook_country, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15", 
                               test_step='summing country % per week & account')

✅ Inner join test FBE_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



In [21]:
facebook_country

,w/c,Channel ID,Channel Name,page_fans_country,PlaceID,Sum_page_fans_country,country_%
0,2025-03-31,FBE237388053065908,BBC Culture,16122,MAL,1234268,0.013062
1,2025-03-31,FBE146214266618,BBC Strictly Come Dancing,4859,NIG,1219009,0.003986
2,2025-03-31,FBE1648071085436082,BBC Rip Off Britain,22,TUR,55848,0.000394
3,2025-03-31,FBE294662213128,BBC News Stories,3776,MOR,692997,0.005449
4,2025-03-31,FBE124158667615790,BBC News Ukrainian,11772,ITA,573511,0.020526
...,...,...,...,...,...,...,...
171715,2025-12-01,FBE236659822607,BBC Hausa,5793,INO,8318077,0.000696
171716,2025-12-01,FBE114577681901041,Later with Jools Holland,250,UAE,194491,0.001285
171717,2025-12-01,FBE114577681901041,Later with Jools Holland,581,FIN,194491,0.002987
171718,2025-12-01,FBE166580710064489,BBC Burmese,13252,BRA,26781329,0.000495


In [22]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
facebook_country[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv",
                        index=None)

In [23]:
facebook_country.sort_values('w/c')['w/c'].unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00']
Length: 37, dtype: datetime64[ns]